In [55]:
import os
import pandas as pd     
import ast

In [56]:

def split_dict_column(df, column_name, prefix=None, drop_original=True):
    """
    Splits a column of dictionary-like strings into separate columns.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): Name of the column containing dictionaries or stringified dictionaries.
        prefix (str, optional): Prefix for the new columns. Defaults to the original column name.
        drop_original (bool, optional): Whether to drop the original column. Defaults to True.

    Returns:
        pd.DataFrame: The updated DataFrame with new columns.
    """
    # Use the original column name as prefix if none provided
    if prefix is None:
        prefix = column_name.replace(":", "").replace(" ", "_")

    # Convert stringified dicts to real dicts
    df[column_name] = df[column_name].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # Expand the dictionary column
    expanded_cols = df[column_name].apply(pd.Series)

    # Rename the new columns with the prefix
    expanded_cols = expanded_cols.rename(columns=lambda k: f"{prefix}_{k}")

    # Combine with the original DataFrame
    df = pd.concat([df, expanded_cols], axis=1)

    # Optionally drop the original column
    if drop_original:
        df = df.drop(columns=[column_name])

    return df

In [57]:
def fix_json_file(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as infile:
        content = infile.read().strip()

    # If it's already a proper JSON array, do nothing and just copy
    if content.startswith('[') and content.endswith(']'):
        with open(output_path, 'w', encoding='utf-8') as outfile:
            outfile.write(content)
        return

    # Remove trailing comma if it exists
    if content.endswith(','):
        content = content[:-1].rstrip()

    # Wrap in brackets to form a valid JSON array
    content = f'[{content}]'

    with open(output_path, 'w', encoding='utf-8') as outfile:
        outfile.write(content)

In [58]:
def get_csv(path):
    #csv_path = path.split(os.sep)[-1].replace('.json','.csv')

    dataset = path.split(os.sep)[-1].split("_")[0]
    model = path.split(os.sep)[-1].split("_")[1]

    fix_json_file(path,path)
    result_file = pd.read_json(os.path.join(path))
    result_file.to_csv("temp.csv", index=False)
    result_file = pd.read_csv("temp.csv")
    
    result_file = split_dict_column(result_file, "LinkTeller unlearn:")
    result_file = split_dict_column(result_file, "LinkTeller original:")
    result_file = result_file.drop(columns=['LinkTeller_unlearn_0', 'parameters', 'RelearnTime', 'dataset'])

    result_file['dataset'] = dataset
    result_file['model'] = model

    return result_file

In [59]:
csv = get_csv(os.path.join("output","runs","LinkAttack", "Citeseer_GIN_edge.json"))

In [60]:
csv

,unlearner,RunTime,AIN,AUS,UMIA,sklearn.metrics.accuracy_score.test.unlearned.on_graph:True,sklearn.metrics.accuracy_score.test.original.on_graph:True,sklearn.metrics.accuracy_score.test.unlearned.on_graph:False,sklearn.metrics.accuracy_score.test.original.on_graph:False,LinkTeller_unlearn_auc,LinkTeller_unlearn_ap,LinkTeller_original_auc,LinkTeller_original_ap,dataset,model
0,GoldModelGraph,0.979264,0.502488,0.896461,0.567167,0.753754,0.752252,0.762763,0.770270,0.390031,0.435658,0.421202,0.442621,Citeseer,GIN
1,UNSIR,0.341477,1.000000,0.931871,0.551220,0.764264,0.746246,0.765766,0.767267,0.424948,0.444084,0.434899,0.448873,Citeseer,GIN
2,SelectiveSynapticDampening,27.773766,1.000000,0.939883,0.551407,0.761261,0.762763,0.768769,0.758258,0.995796,0.997326,0.401927,0.447689,Citeseer,GIN
3,Scrub,27.651800,0.401198,0.966606,0.545779,0.770270,0.758258,0.762763,0.761261,0.404187,0.423789,0.387314,0.430672,Citeseer,GIN
4,Cascade,35.463915,1.000000,0.929762,0.549156,0.756757,0.761261,0.759760,0.761261,0.442706,0.445536,0.379965,0.418998,Citeseer,GIN
5,FisherForgetting,30.127856,1.000000,0.930671,0.544278,0.749249,0.762763,0.762763,0.764264,0.995925,0.997326,0.447625,0.468655,Citeseer,GIN
6,SuccessiveRandomLabels,27.587344,1.000000,0.938471,0.554784,0.759760,0.758258,0.765766,0.765766,0.384798,0.422995,0.407904,0.427646,Citeseer,GIN
7,eu_k,0.205669,1.000000,1.026041,0.524578,0.828829,0.743243,0.815315,0.756757,0.390517,0.418063,0.354800,0.401696,Citeseer,GIN
8,Finetuning,0.119828,1.000000,0.940128,0.532458,0.753754,0.747748,0.755255,0.762763,0.401641,0.436123,0.420487,0.451109,Citeseer,GIN
9,NegGrad,0.103432,101.000000,0.944307,0.543902,0.758258,0.756757,0.770270,0.762763,0.393835,0.428586,0.405616,0.439303,Citeseer,GIN
